# Sent Below — ML/AI Technical Demo

**Adaptive Game AI with Deep Reinforcement Learning**

This notebook demonstrates the ML systems powering *Sent Below*, a dungeon crawler with real-time adaptive AI. It covers:

1. **Dueling DQN with Self-Attention** — Enemy decision-making via deep RL
2. **Neural Player Modeling** — Dynamic difficulty adjustment
3. **Content Recommendation** — ML-driven loot and room selection
4. **Training Pipeline** — Offline training with TensorBoard integration
5. **Model Serving** — Production deployment via FastAPI + Docker
6. **A/B Testing** — Statistical model comparison framework
7. **Performance Benchmarking** — Real-time inference latency analysis

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn
import numpy as np
import json
import time
from collections import defaultdict

print(f'PyTorch {torch.__version__}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

## 1. Enemy AI Architecture — Dueling DQN with Self-Attention

The enemy AI uses a **Dueling DQN** architecture enhanced with **self-attention** over the state features.

### Why Dueling DQN?
Standard DQN learns Q(s,a) directly. Dueling DQN decomposes this into:
- **V(s)**: How good is this state? (value stream)
- **A(s,a)**: How much better is each action? (advantage stream)
- **Q(s,a) = V(s) + A(s,a) - mean(A)**

This separation enables faster learning because the value stream can generalize across actions.

### Why Self-Attention?
Different game situations require attending to different state features. When an enemy is low HP, the HP feature should dominate decision-making. Self-attention learns these dynamic feature importances.

In [ ]:
from ai.enemy_ai import EnemyBrain, EnemyNetwork, ACTIONS, STATE_DIM, NUM_ACTIONS

brain = EnemyBrain(epsilon_start=0.3)
net = brain.policy_net

# Model architecture summary
print('=== Dueling DQN Architecture ===')
print(f'State dim: {STATE_DIM} | Action space: {NUM_ACTIONS} ({ACTIONS})')
print(f'Total parameters: {sum(p.numel() for p in net.parameters()):,}')
print()
for name, module in net.named_modules():
    if isinstance(module, (nn.Linear, nn.BatchNorm1d)):
        params = sum(p.numel() for p in module.parameters())
        print(f'  {name:<30} {module.__class__.__name__:<15} params={params}')

In [ ]:
# Demonstrate self-attention feature weighting
print('=== Self-Attention Feature Gating ===')
print('Low HP enemy state → attention should upweight HP and flee-related features')
print()

low_hp_state = torch.tensor([[0.1, 0.8, 0.3, 1.0, 0.5, 0.5, 1.0, 0.0, 0.0, 0.5]])
high_hp_state = torch.tensor([[0.9, 0.8, 0.3, 1.0, 0.5, 0.5, 1.0, 0.0, 0.0, 0.5]])

net.eval()
with torch.no_grad():
    low_q = net(low_hp_state).squeeze()
    high_q = net(high_hp_state).squeeze()

feature_names = ['enemy_hp', 'player_hp', 'distance', 'atk_ready',
                 'strength', 'defense', 'aggressive', 'defensive', 'support', 'time']

print(f'{"Action":<15} {"Low HP Q":>10} {"High HP Q":>10} {"Δ":>10}')
print('-' * 47)
for i, action in enumerate(ACTIONS):
    print(f'{action:<15} {low_q[i]:>10.3f} {high_q[i]:>10.3f} {high_q[i]-low_q[i]:>10.3f}')

print(f'\nLow HP best action: {ACTIONS[low_q.argmax()]}')
print(f'High HP best action: {ACTIONS[high_q.argmax()]}')

## 2. Training Loop — Experience Replay + Reward Shaping

The DQN trains through simulated combat episodes with:
- **Experience replay buffer** (10K capacity) for breaking temporal correlations
- **Target network** with soft Polyak updates (τ=0.01)
- **Epsilon-greedy** exploration with exponential decay
- **Multi-objective reward shaping**: damage dealt, damage taken, tactical positioning

In [ ]:
from training.train_pipeline import SimulatedCombatEnv

# Train for a few episodes and track metrics
brain = EnemyBrain(epsilon_start=1.0, epsilon_decay=0.995)
env = SimulatedCombatEnv()

episode_rewards = []
losses = []
epsilons = []

for ep in range(100):
    behavior = ['aggressive', 'defensive', 'support'][ep % 3]
    state = env.reset(behavior)
    done = False
    ep_reward = 0
    
    while not done:
        action = brain.decide_action(state, behavior)
        action_idx = ACTIONS.index(action)
        next_state, reward, done, info = env.step(action_idx)
        brain.store_experience(state, action_idx, reward, next_state, done)
        ep_reward += reward
        state = next_state
        
        if brain.total_steps % 4 == 0:
            loss = brain.train_step()
            if loss is not None:
                losses.append(loss)
        if brain.total_steps % 10 == 0:
            brain.update_target_network()
    
    episode_rewards.append(ep_reward)
    epsilons.append(brain.epsilon)

# Print training curve
print('=== Training Curve (100 episodes) ===')
window = 10
for i in range(0, len(episode_rewards), window):
    chunk = episode_rewards[i:i+window]
    avg = np.mean(chunk)
    bar = '█' * int(max(0, (avg + 5) * 3))
    print(f'  Episodes {i+1:>3d}-{i+window:>3d}: reward={avg:+.2f} eps={epsilons[min(i+window-1, len(epsilons)-1)]:.3f} {bar}')

print(f'\nFinal loss: {losses[-1]:.4f}' if losses else 'No training steps')
print(f'Replay buffer: {len(brain.replay_buffer)} experiences')
print(f'Training steps: {brain.training_steps}')

## 3. Dynamic Difficulty Adjustment (DDA)

A neural **player model** predicts survival probability and enjoyment, then a PID-style controller adjusts enemy scaling to keep the player in the **flow zone** (~60% survival).

In [ ]:
from ai.director import AIDirector

director = AIDirector(update_interval=0.5)

# Simulate a player who starts strong and gradually struggles
print('=== DDA Difficulty Curve ===')
print(f'{"Game Time":>10} {"HP%":>6} {"Kills/min":>10} {"Deaths":>7} {"Difficulty":>11} {"Predicted Surv":>15}')
print('-' * 65)

game_time = 0.0
for step in range(20):
    # Simulate degrading player performance
    hp_pct = max(0.1, 0.95 - step * 0.04)
    kills_per_min = max(0.5, 5.0 - step * 0.2)
    deaths = min(8, step // 3)
    
    player_stats = {
        'hp_pct': hp_pct, 'mp_pct': 0.6, 'level': 5,
        'kills_per_min': kills_per_min, 'deaths': deaths,
        'damage_ratio': max(0.3, 1.5 - step * 0.06),
        'ability_usage': 10, 'potion_usage': step * 0.3,
        'clear_time_ratio': min(2.0, 0.8 + step * 0.05),
    }
    
    director.update(player_stats, floor_num=3, game_time=game_time, dt=0.5)
    stats = director.get_stats()
    diff = stats['difficulty_modifier']
    surv = stats.get('predicted_survival', 0.6)
    
    bar = '▓' * int(diff * 10)
    print(f'{game_time:>8.0f}s {hp_pct:>5.0%} {kills_per_min:>9.1f} {deaths:>6d} {diff:>10.3f}x {surv:>14.1%} {bar}')
    
    game_time += 15
    director.record_event('kill', {}, game_time)
    if step % 3 == 0:
        director.record_event('death', {}, game_time)

## 4. Data Pipeline — From Gameplay to Training

The data pipeline captures gameplay events in structured JSON-Lines format, processes them into ML-ready feature tensors, and provides PyTorch Dataset/DataLoader integration.

In [ ]:
from ai.data_pipeline import EventLogger, DataProcessor, CombatTransitionDataset, create_data_loaders
import shutil

# Simulate gameplay data collection
logger = EventLogger(log_dir='demo_logs', buffer_size=50)

env = SimulatedCombatEnv()
for ep in range(20):
    state = env.reset('aggressive')
    done = False
    while not done:
        action_idx = np.random.randint(NUM_ACTIONS)
        next_state, reward, done, info = env.step(action_idx)
        logger.log_combat_step(
            state, ACTIONS[action_idx], reward, next_state, done,
            'aggressive', ep * 10.0
        )
        state = next_state
logger.close()

# Process into dataset
processor = DataProcessor(log_dir='demo_logs')
states, actions, rewards, next_states, dones = processor.extract_combat_transitions()
print(f'Collected {len(states)} transitions from {logger.event_count} events')

# Create PyTorch DataLoader
dataset = CombatTransitionDataset(states, actions, rewards, next_states, dones)
train_loader, val_loader, test_loader = create_data_loaders(dataset, batch_size=32)

batch = next(iter(train_loader))
print(f'\nDataLoader batch shapes:')
for name, tensor in zip(['states', 'actions', 'rewards', 'next_states', 'dones'], batch):
    print(f'  {name}: {tensor.shape}')

# Session summary
summaries = processor.compute_session_summary()
print(f'\nSession summary: {json.dumps(summaries[0], indent=2)}')

shutil.rmtree('demo_logs', ignore_errors=True)

## 5. A/B Testing — Statistical Model Comparison

Before deploying a new model version, we run A/B tests to validate improvement with statistical rigor:
- Welch's t-test for significance
- Cohen's d for effect size
- Bootstrap confidence intervals

In [ ]:
from ai.ab_testing import ABTestRunner, Experiment, Variant

experiment = Experiment(
    name='DQN v1 (MLP) vs v2 (Dueling+Attention)',
    variants=[
        Variant('v1_mlp', 'models/v1.pt', 0.5, 'Baseline 3-layer MLP'),
        Variant('v2_dueling', 'models/v2.pt', 0.5, 'Dueling DQN + Self-Attention'),
    ],
    metric='avg_reward',
    min_sessions_per_variant=20,
)

runner = ABTestRunner(experiment)
rng = np.random.default_rng(42)

# Simulate 50 sessions (v2 is ~15% better)
for i in range(50):
    variant = runner.assign_session(f'session_{i}')
    base = 1.8 if variant.name == 'v2_dueling' else 1.5
    runner.record_session(f'session_{i}', variant.name, {
        'avg_reward': base + rng.normal(0, 0.4),
        'win_rate': 0.55 + (0.08 if variant.name == 'v2_dueling' else 0) + rng.normal(0, 0.1),
    })

report = runner.analyze()

print('=== A/B Test Results ===')
print(f'Experiment: {report["experiment"]}')
print(f'Metric: {report["metric"]}\n')

for vname, vstats in report['variants'].items():
    print(f'{vname}:')
    print(f'  Mean: {vstats["mean"]:.3f} ± {vstats["std"]:.3f}')
    print(f'  95% CI: [{vstats["ci_95"][0]:.3f}, {vstats["ci_95"][1]:.3f}]')
    print(f'  Sessions: {vstats["sessions"]}\n')

comp = report['comparisons'][0]
print(f'Statistical Test:')
print(f'  p-value: {comp["welch_t_test"]["p_value"]:.4f}')
print(f'  Cohen\'s d: {comp["cohens_d"]:.3f} ({comp["effect_size"]})')
print(f'  Significant: {comp["significant_at_005"]}')
print(f'  Winner: {comp["winner"]}')
print(f'\n→ Recommendation: Deploy {report["recommendation"]}')

## 6. Performance Benchmarking

Real-time games have strict frame budgets. At 60fps, the entire frame must complete in 16.7ms. We measure AI inference cost to ensure it fits within budget.

In [ ]:
from training.benchmark import measure_latency, measure_throughput, measure_memory

brain = EnemyBrain()
net = brain.policy_net
net.eval()

# Latency
single_input = torch.randn(1, STATE_DIM)
latency = measure_latency(net, single_input, warmup=50, iterations=500)
print('=== Inference Latency ===')
print(f'  Mean:  {latency["mean_ms"]:.3f}ms')
print(f'  P95:   {latency["p95_ms"]:.3f}ms')
print(f'  P99:   {latency["p99_ms"]:.3f}ms')

# Throughput across batch sizes
print('\n=== Throughput ===')
throughput = measure_throughput(net, STATE_DIM, [1, 8, 32, 64], duration_sec=1.0)
for bs, tp in throughput.items():
    print(f'  Batch {bs:>3}: {tp["inferences_per_sec"]:>10,.0f} inf/sec')

# Frame budget
frame_ms = 1000 / 60
pct = (latency['p95_ms'] / frame_ms) * 100
print(f'\n=== Frame Budget (60fps = {frame_ms:.1f}ms) ===')
print(f'  AI cost per enemy: {latency["p95_ms"]:.3f}ms ({pct:.1f}% of frame)')
print(f'  Max enemies (single-threaded): ~{int(frame_ms / latency["p95_ms"])}')
if 32 in throughput:
    batch_ms = throughput[32]['latency_per_batch_ms']
    print(f'  With batch-32 inference: ~{int(frame_ms / batch_ms) * 32} enemies')

## 7. Deployment Architecture

The system uses AWS for cloud-backed model storage and optional container deployment:

```
┌─────────────────────────────────────────────────────────┐
│                    CI/CD Pipeline                        │
│  GitHub Actions → Test → Train → Benchmark → Docker    │
└──────────────┬──────────────────────────┬───────────────┘
               │                          │
     ┌─────────▼─────────┐      ┌────────▼────────┐
     │  Local Training    │      │  Amazon S3       │
     │  - TensorBoard     │      │  - .pt / .onnx   │
     │  - Checkpointing   │ ──── │  - Versioned     │
     │  - ONNX export     │      │    artefacts     │
     └───────────────────┘      └────────┬────────┘
                                         │
                                ┌────────▼────────┐
                                │  DynamoDB        │
                                │  Model Registry  │
                                │  - version       │
                                │  - metrics       │
                                │  - s3_prefix     │
                                └────────┬────────┘
                                         │
                              ┌──────────▼──────────┐
                              │  AWS ECS Fargate     │
                              │  (FastAPI serve)     │
                              │  - S3 model download │
                              │  - Health checks     │
                              │  - CloudWatch logs   │
                              └──────────┬──────────┘
                                         │
                          ┌──────────────▼──────────────┐
                          │   Game Client                │
                          │   - Real-time inference      │
                          │   - A/B test routing         │
                          │   - Gameplay data logging    │
                          └─────────────────────────────┘
```

### AWS integration:
- **S3** — Versioned model artefact storage (.pt, .onnx, scaler)
- **DynamoDB** — Lightweight model registry (name + version + eval metrics)
- **ECS Fargate** — Optional containerised deployment for FastAPI serving
- **CloudWatch** — Log aggregation from ECS tasks

### Other infrastructure:
- **Multi-stage Docker** build (train / serve / game targets)
- **FastAPI** model serving with batch inference, health checks, metrics
- **TensorBoard** for experiment tracking and visualisation
- **A/B testing** with statistical significance analysis
- **ONNX export** for cross-platform model deployment

## Summary

| Component | Technology | Purpose |
|-----------|-----------|--------|
| Enemy AI | PyTorch Dueling DQN + Self-Attention | Real-time adaptive enemy behaviour |
| Player Modelling | PyTorch Neural Network | Dynamic difficulty adjustment |
| Content Rec. | Embedding dot-product + softmax | Intelligent loot/room selection |
| Training | PyTorch + TensorBoard + LR Scheduling | Offline model training |
| Serving | FastAPI + Uvicorn | Production REST inference API |
| Model Storage | Amazon S3 | Versioned .pt / .onnx artefacts |
| Model Registry | Amazon DynamoDB | Version tracking + eval metrics |
| Containerisation | Docker multi-stage | Reproducible build and deploy |
| Cloud Deploy | AWS ECS Fargate | Scalable ML serving |
| CI/CD | GitHub Actions | Automated test -> train -> deploy |
| A/B Testing | Statistical framework | Model comparison and rollout |
| Data Pipeline | JSON-Lines + PyTorch Dataset | End-to-end ML lifecycle |
| Benchmarking | Latency/throughput profiler | Performance regression detection |